# UAI 735I - Bash examples

# Ubuntu Command Line & Bash — Practical Lecture

**Duration:** ~90–120 minutes
**Audience:** 2nd–3rd year Bc. students (Applied Informatics), comfortable with C++/OOP, new to Linux CLI.
**Format:** Live demo-heavy lecture. Every section has runnable examples.
**Prerequisites:** Ubuntu 22.04/24.04 (or WSL2), a terminal, a user account with `sudo`.

---

## 0. Agenda (2 min)

1. Why the shell matters
2. Navigating the filesystem
3. Files & directories: create, copy, move, delete
4. Viewing & editing files
5. Pipes, redirection, and filters
6. Permissions & ownership
7. Processes & job control
8. Environment variables & PATH
9. Bash scripting basics
10. Useful one-liners & wrap-up

> **Teaching tip:** Open two terminals side by side — one for typing, one for `man` pages / docs.

---

## 1. Why the Shell Matters (5 min)

- Servers, containers, CI/CD, HPC, embedded — no GUI.
- Composable: small tools chained together do big things.
- Scriptable: automate anything you do twice.
- Faster than clicking once you know ~20 commands.

**Mental model:** the shell is a REPL for your OS. Every command is a program. The shell's job is to *parse*, *expand*, and *execute*.

---

## 2. Navigating the Filesystem (10 min)

```bash
pwd                     # where am I?
ls                      # list
ls -la                  # long + hidden
ls -lh /var/log         # human-readable sizes
cd /etc                 # absolute path
cd ..                   # up one
cd ~                    # home
cd -                    # previous directory (toggle)
tree -L 2               # if installed: apt install tree
```

**Special paths:**
| Symbol | Meaning |
|---|---|
| `.` | current dir |
| `..` | parent dir |
| `~` | home |
| `/` | root |
| `-` | previous dir (for `cd`) |

**Exercise 1:** From `/`, navigate to `/var/log` using only relative moves after one `cd`. Then `cd -` back.

---

## 3. Files & Directories (15 min)

```bash
mkdir demo && cd demo
mkdir -p a/b/c                  # nested, no error if exists

touch file1.txt file2.txt
echo "hello" > file1.txt        # write (overwrite)
echo "world" >> file1.txt       # append

cp file1.txt copy.txt
cp -r a/ a_backup/              # recursive
mv copy.txt renamed.txt
mv renamed.txt a/               # move into dir
rm file2.txt
rm -r a_backup                  # recursive delete
rm -rf a_backup                 # force (DANGEROUS)
```

**Safety habits:**
- Never `rm -rf` with a variable you didn't `echo` first.
- `rm -i` asks before deleting.
- Trash isn't automatic — deleted is deleted.

**Globbing (shell expands these, not the command):**
```bash
ls *.txt
ls file?.txt
ls [abc]*.txt
ls **/*.py          # with globstar: shopt -s globstar
```

**Exercise 2:** Create `lab/` with subdirs `src/`, `docs/`, `tests/`. Put 3 `.py` files in `src/`, 2 `.md` in `docs/`. Then move all `.md` into `src/` with one command.

---

## 4. Viewing & Editing Files (10 min)

```bash
cat file1.txt
less file1.txt          # q to quit, / to search
head -n 5 file1.txt
tail -n 5 file1.txt
tail -f /var/log/syslog # follow (Ctrl+C to stop)

wc -l file1.txt         # lines
file file1.txt          # type detection
stat file1.txt          # metadata
```

**Editing:**
- `nano` — beginner-friendly: `nano file.txt`, `Ctrl+O` save, `Ctrl+X` exit.
- `vim` — `vimtutor` (run it once, it's 30 min and pays off forever).
- VS Code / editors via `code .` if installed.

**Exercise 3:** Create a 100-line file with `seq 1 100 > nums.txt`. Show lines 40–50 with `sed -n '40,50p' nums.txt` (preview of filters).

---

## 5. Pipes, Redirection & Filters (25 min — the core)

### 5.1 Redirection

```bash
command > file        # stdout to file (overwrite)
command >> file       # append
command 2> err.log    # stderr to file
command &> all.log    # both (bash)
command < input.txt   # stdin from file
```

### 5.2 Pipes

`|` connects stdout of left to stdin of right.

```bash
ls -l | wc -l                    # count files
cat /etc/passwd | grep bash      # users with bash
ps aux | grep python | head
```

### 5.3 The essential filters

```bash
# grep — search
grep "error" app.log
grep -i "error" app.log          # case-insensitive
grep -r "TODO" src/              # recursive
grep -v "debug" app.log          # invert
grep -E "err(or)?|warn" app.log  # extended regex

# sed — stream edit
sed 's/foo/bar/' file            # first per line
sed 's/foo/bar/g' file           # global
sed -n '10,20p' file             # print range
sed '/^#/d' file                 # delete comment lines

# awk — field processing
awk '{print $1}' file            # first column
awk -F: '{print $1, $7}' /etc/passwd
awk '$3 > 1000 {print $1}' /etc/passwd

# sort / uniq
sort file | uniq -c | sort -rn | head

# cut
cut -d: -f1 /etc/passwd

# tr
echo "HELLO" | tr 'A-Z' 'a-z'

# xargs — build command lines
find . -name "*.tmp" | xargs rm
find . -name "*.py" -print0 | xargs -0 wc -l
```

### 5.4 The classic pipeline

**Top 5 most frequent commands in your shell history:**
```bash
history | awk '{print $2}' | sort | uniq -c | sort -rn | head -5
```

**Largest files in a directory tree:**
```bash
du -ah . | sort -rh | head -10
```

**Find and count Python lines per file:**
```bash
find . -name "*.py" -exec wc -l {} + | sort -n
```

**Live log error counter:**
```bash
tail -f app.log | grep --line-buffered "ERROR" | awk '{print $1, $2}' | uniq -c
```

**Exercise 4:** From `/etc/passwd`, print usernames sorted alphabetically whose shell is `/bin/bash`, one per line, without the shell column.

**Exercise 5:** Given `access.log` (Apache format), produce a list of top 10 IPs by request count. *(Hint: `awk '{print $1}' | sort | uniq -c | sort -rn | head`)*

---

## 6. Permissions & Ownership (10 min)

```bash
ls -l
# -rw-r--r-- 1 alice users 1234 Sep 16 10:00 file.txt
#  ^^^        owner group
#  rwx for owner, group, others
```

**chmod — symbolic and octal:**
```bash
chmod +x script.sh          # add execute
chmod u+w file              # owner write
chmod go-r file             # remove read from group+others
chmod 644 file              # rw-r--r--
chmod 755 script.sh         # rwxr-xr-x
chmod 600 secret.key        # rw-------
```

| Octal | Perms |
|---|---|
| 7 | rwx |
| 6 | rw- |
| 5 | r-x |
| 4 | r-- |
| 0 | --- |

**chown / chgrp (needs sudo):**
```bash
sudo chown alice:users file
sudo chown -R alice:users /opt/app
```

**umask** — default permission mask for new files:
```bash
umask            # e.g. 022 → new files 644, dirs 755
```

**Exercise 6:** Create `run.sh`, make it executable only by you (700), run it, then change it to 755.

---

## 7. Processes & Job Control (10 min)

```bash
ps aux                  # all processes
ps aux | grep python
top                     # interactive (q to quit)
htop                    # nicer, if installed

kill PID                # SIGTERM
kill -9 PID             # SIGKILL (last resort)
pkill -f "python app"   # by pattern
killall firefox
```

**Jobs in the current shell:**
```bash
sleep 300 &             # background
jobs                    # list
fg %1                   # bring to foreground
bg %1                   # resume in background
Ctrl+Z                  # suspend current
Ctrl+C                  # interrupt
```

**nohup / disown — survive logout:**
```bash
nohup python train.py > train.log 2>&1 &
```

**Exercise 7:** Start `sleep 1000` in background, list jobs, kill it by PID, verify it's gone.

---

## 8. Environment Variables & PATH (10 min)

```bash
echo $HOME
echo $PATH
env                     # all vars
export MY_VAR="hello"
unset MY_VAR

# PATH manipulation
export PATH="$HOME/bin:$PATH"
```

**Persist them:** in `~/.bashrc` (interactive shells), `~/.profile` (login), or `/etc/environment` (system-wide).

```bash
echo 'export PATH="$HOME/bin:$PATH"' >> ~/.bashrc
source ~/.bashrc        # reload
```

**`which` vs `type` vs `command -v`:**
```bash
which python3           # path
type ls                 # alias/builtin/binary?
command -v git
```

**Exercise 8:** Create `~/bin/hello` that echoes "hi", `chmod +x` it, add `~/bin` to PATH permanently, open a new terminal, run `hello`.

---

## 9. Bash Scripting Basics (20 min)

### 9.1 Shebang & structure

```bash
#!/usr/bin/env bash
set -euo pipefail           # exit on error, undefined var, pipe fail
IFS=$'\n\t'
```

Always use `set -euo pipefail` in real scripts — it turns silent bugs into loud failures.

### 9.2 Variables & quoting

```bash
name="World"
echo "Hello, $name"
echo 'Hello, $name'         # literal
echo "Files: $(ls | wc -l)" # command substitution
echo "Today: $(date +%F)"

# Defaults
: "${PORT:=8080}"           # set if unset
echo "${PORT}"
```

**Rule:** always quote `"$var"` unless you specifically want word-splitting.

### 9.3 Conditionals

```bash
if [[ -f "$file" ]]; then
  echo "file exists"
elif [[ -d "$file" ]]; then
  echo "it's a dir"
else
  echo "missing"
fi

# Test operators
[[ -z "$s" ]]     # empty string
[[ -n "$s" ]]     # non-empty
[[ "$a" == "$b" ]]
[[ "$a" =~ ^[0-9]+$ ]]   # regex
(( n > 10 ))             # arithmetic
```

### 9.4 Loops

```bash
for f in *.txt; do
  echo "Processing $f"
done

for i in {1..5}; do echo $i; done

while read -r line; do
  echo ">> $line"
done < input.txt
```

### 9.5 Functions

```bash
log() {
  echo "[$(date +%T)] $*"
}

greet() {
  local name="$1"
  echo "Hi, $name"
}

greet "Alice"
log "done"
```

### 9.6 Arguments & exit codes

```bash
#!/usr/bin/env bash
set -euo pipefail

if [[ $# -lt 1 ]]; then
  echo "Usage: $0 <directory>" >&2
  exit 1
fi

dir="$1"
[[ -d "$dir" ]] || { echo "Not a directory" >&2; exit 2; }

count=$(find "$dir" -type f | wc -l)
echo "Files in $dir: $count"
exit 0
```

| Code | Meaning |
|---|---|
| 0 | success |
| 1 | general error |
| 2 | misuse |
| 126/127 | not executable / not found |
| 130 | Ctrl+C |

Check `$?` after a command:
```bash
ls /nonexistent
echo $?         # 2
```

### 9.7 Full example: backup script

```bash
#!/usr/bin/env bash
set -euo pipefail

SRC="${1:?source dir required}"
DEST="${2:-$HOME/backups}"
STAMP=$(date +%Y%m%d-%H%M%S)
ARCHIVE="$DEST/backup-$STAMP.tar.gz"

mkdir -p "$DEST"
tar -czf "$ARCHIVE" -C "$SRC" .
echo "Created $ARCHIVE ($(du -h "$ARCHIVE" | cut -f1))"

# keep only last 7
ls -1t "$DEST"/backup-*.tar.gz | tail -n +8 | xargs -r rm --
```

**Exercise 9:** Write `count.sh` that takes a directory and prints the number of `.py` files and total lines. Handle missing argument with a usage message and exit 1.

**Exercise 10:** Write `watch.sh` that every 2 seconds prints the current time and number of running `python` processes; stop on Ctrl+C (use `trap`).

```bash
trap 'echo "bye"; exit 0' INT
while true; do
  printf '%s python procs: %s\n' "$(date +%T)" "$(pgrep -c python || echo 0)"
  sleep 2
done
```

---

## 10. Useful One-Liners & Wrap-Up (5 min)

```bash
# Replace text in all files
grep -rl "old" . | xargs sed -i 's/old/new/g'

# Find files modified in last 24h
find . -type f -mtime -1

# Ports in use
ss -tulpn | grep LISTEN

# Disk usage
df -h
du -sh *

# Download
curl -O https://example.com/file.zip
wget -c https://example.com/big.iso

# JSON pretty print
curl -s https://api.github.com/repos/torvalds/linux | jq '.stargazers_count'

# Watch a command every 2s
watch -n 2 'df -h'
```

**Help yourself:**
```bash
man ls
ls --help
tldr ls            # if installed: apt install tldr
apropos "copy file"
```

**Cheat sheet to remember:**
- `Ctrl+R` — reverse search history
- `Ctrl+A` / `Ctrl+E` — start / end of line
- `Ctrl+U` / `Ctrl+K` — delete to start / end
- `!!` — last command, `sudo !!` is gold
- `Tab` — completion, `Tab Tab` — list options

---

## Homework (optional)

1. Write a script `sysinfo.sh` that prints: hostname, kernel, uptime, disk usage of `/`, top 3 memory-consuming processes.
2. Parse an Apache `access.log` (generate one or use a sample) and produce: total requests, top 5 IPs, top 5 URLs, count of 404s.
3. Automate: a script that renames all `.jpeg` files in a directory to `.jpg` (dry-run first with `echo`, then real).

---

## Instructor Notes

- **Pacing:** Sections 5 and 9 are the meat — don't rush them. Sections 2–4 can be fast if the room is comfortable.
- **Live coding > slides.** Type every command; let students predict output.
- **Break at ~60 min** if the slot is 2h.
- **Common student traps:**
  - Forgetting to quote variables.
  - Using `>` when they meant `>>`.
  - Running `rm -rf $DIR/` when `$DIR` is empty.
  - Editing `.bashrc` and not sourcing it.
- **Assessment tie-in:** the semester projects all require CLI + Git + scripting — this lecture is the foundation.

---

## Quick Reference Card

| Task | Command |
|---|---|
| Where am I | `pwd` |
| List all | `ls -la` |
| Make dirs | `mkdir -p a/b/c` |
| Copy tree | `cp -r src dst` |
| Find files | `find . -name "*.py"` |
| Search text | `grep -rn "TODO" .` |
| Replace text | `sed -i 's/a/b/g' file` |
| Columns | `awk '{print $1}'` |
| Sort + count | `sort | uniq -c | sort -rn` |
| Permissions | `chmod 755 f` |
| Kill process | `pkill -f name` |
| Env var | `export X=1` |
| Script header | `#!/usr/bin/env bash` + `set -euo pipefail` |